# Estatísticas por Copa e por seleção

## Task 3: calcular as estatísticas por Copa e por seleção

## O que a transmissão precisa

1. **Por Copa:** partidas, gols, média de gols por partida e público médio.
2. **Por seleção:** jogos, vitórias, gols marcados, gols sofridos e saldo.

In [ ]:
# a sessão Spark deste notebook: já vem pronta (é o código do vídeo 1.4, com outro appName)
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-estatisticas")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

A célula abaixo já vem pronta: é o mesmo código dos vídeos 2.1 (leitura e limpeza) e 2.2 (padronização), na mesma
ordem.

In [ ]:
# vídeo 2.1: leitura e limpeza
from wc_pipeline.schemas import MATCHES

partidas_brutas = (
    spark.read
    .option("header", "true")
    .schema(MATCHES)
    .csv(f"{RAW}/WorldCupMatches.csv")
)

partidas_limpas = (
    partidas_brutas
    .filter(F.col("MatchID").isNotNull())
    .dropDuplicates(["MatchID"])
)

# vídeo 2.2: padronização
jogos = partidas_limpas.select(
    F.col("Year").alias("ano"),
    F.col("MatchID").alias("match_id"),
    F.col("Stage").alias("fase"),
    F.col("Home Team Name").alias("mandante"),
    F.col("Home Team Goals").alias("gols_mandante"),
    F.col("Away Team Goals").alias("gols_visitante"),
    F.col("Away Team Name").alias("visitante"),
    F.col("Attendance").alias("publico_texto"),
    F.col("Datetime").alias("data_texto"),
)

FORMATO_DATA = "dd MMM yyyy - HH:mm"

partidas = (
    jogos
    .withColumn("publico", F.regexp_replace("publico_texto", ",", "").try_cast("int"))
    .withColumn("data_partida", F.try_to_timestamp(F.col("data_texto"), F.lit(FORMATO_DATA)))
    .withColumn("total_gols", F.col("gols_mandante") + F.col("gols_visitante"))
    .withColumn(
        "resultado",
        F.when(F.col("gols_mandante") > F.col("gols_visitante"), "mandante")
         .when(F.col("gols_mandante") < F.col("gols_visitante"), "visitante")
         .otherwise("empate"),
    )
    .drop("publico_texto", "data_texto")
)

partidas.printSchema()

## Agregar sem agrupar: o histórico inteiro

O resumo de todas as Copas, a referência para comparar cada Copa com a média histórica. As funções de agregação
(`F.count`, `F.sum`, `F.avg`, `F.max`...) dentro de um `select` resumem a tabela numa linha.

562 partidas, 2.542 gols, média de 4,52 gols por partida e maior placar com 9 gols.

## groupBy + agg: uma linha por Copa

`groupBy` separa os grupos; `agg` diz o que calcular em cada um.

`avg`, `sum` e as outras agregações **ignoram nulos**; só o `count("*")` conta todas as linhas.

## Por seleção: o truque de virar a partida

Cada partida tem duas seleções, em colunas diferentes. Mudamos o **grão** (o que cada linha representa): cada partida
vira duas linhas, uma por seleção.

```
partida:   Brazil  3 x 1  Mexico

vira:      selecao   gols_pro   gols_contra
           Brazil       3           1
           Mexico       1           3
```

O dobro de linhas. `union` empilha **pela posição**; `unionByName`, **pelo nome**, que é o mais seguro.

`F.sum(F.when(...))` **conta com condição**: cada vitória vale 1, o resto 0.

## Conferindo o resultado

O total de gols tem que ser o mesmo pelos dois caminhos: por Copa e por seleção.

## Resultado da task 3

As tabelas por Copa e por seleção estão prontas e batem no total de gols. Próxima task: os joins.

## Para consultar depois

- **`groupBy("ano").count()`** é um atalho que só conta, com a coluna chamada `count`. O `agg` permite várias
  métricas, com os nomes que você escolher.
- **Outras agregações úteis:** `F.countDistinct("estadio")` (quantos valores diferentes), `F.min`, `F.max` e
  `F.stddev`. `F.collect_set` e `F.collect_list` juntam os valores numa lista; cuidado em dado grande, porque uma chave
  com milhões de linhas vira uma lista gigante numa linha só.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()